# calculate annual means 
- input: final dataframe for ICON full (from nbook 62_) 
- moving block bootstrap
- for each bootstrap sample:
    - rebuild the full year with blocks of 50 days drawn random with replacement
    - calculate annual mean
    - for domain, cell features, anvil features and linked cells
- Output: (1 dataset per cell features/anvilfeatures/cells/domain), saved in `output/annual/data`
    - bootstrap samples with annual means
    - mean across bootstraps

In [1]:
import os 
import pickle
import numpy as np
import xarray as xr

import pandas as pd
from pathlib import Path

import sys

sys.path.append("../tools")
from defs_application import icon_years

from cell_props import compute_annual_cell_stats # function for the annual stats

In [2]:
years = icon_years
# inputdir
inputdir = Path("../output/final_dataframes/FULL")


# output directory for data 
output_directory = Path("../output/annual/data")
output_directory.mkdir(parents=True, exist_ok=True)


In [11]:
datasets = ["cell_tracks", "cell_features", "anvil_features", "domain"]

Tsmean = {}
Tsci_low = {}
Tsci_high = {}
Tsboot ={}

for dataset in datasets:

    # import
    data = {}

    for year in years:
        data[year] = pd.read_hdf(os.path.join( inputdir,f"{dataset}_ICON_{year}.h5" ), key="table")

    # compute
    ds_annual = compute_annual_cell_stats( data,block_length_days=50, n_boot=1000, return_boot = True )

    # assign T to anvil features
    Tsmean[dataset] = ds_annual["mean"][
        [v for v in ds_annual["mean"].data_vars if "tas" in v]]

    Tsci_low[dataset] = ds_annual["ci_low"][
        [v for v in ds_annual["ci_low"].data_vars if "tas" in v]]

    Tsci_high[dataset] = ds_annual["ci_high"][
        [v for v in ds_annual["ci_high"].data_vars if "tas" in v]]
   
    Tsboot[dataset] = ds_annual["boot"][
    [v for v in ds_annual["boot"].data_vars if "tas" in v]]

    if dataset == "anvil_features" or dataset == "domain":

        ds_annual["mean"]["tas_mean"] = (Tsmean["cell_features"]["tas_cell"] )

        ds_annual["ci_low"]["tas_mean"] = (Tsci_low["cell_features"]["tas_cell"])

        ds_annual["ci_high"]["tas_mean"] = (Tsci_high["cell_features"]["tas_cell"])
        
        ds_annual["boot"]["tas_mean"] = (Tsboot["cell_features"]["tas_cell"])


    # save
    filepath = os.path.join(
        output_directory,
        f"yearly_stats_abso_boot_{dataset}.pkl")

    with open(filepath, "wb") as f:
        pickle.dump(ds_annual, f)

print(ds_annual["mean"].data_vars)


KeyboardInterrupt


KeyboardInterrupt



In [ ]:
print(ds_annual["boot"].data_vars)